# Stage 7 — Final Model, Submission & Writeup (rebuilt)

Self-contained: rebuilds everything from the raw new files, trains the final augmented ensemble on **all** data, predicts the corrupted test set, writes a **format-validated** submission, and drafts the trustworthiness sections.

In [1]:
import numpy as np, pandas as pd, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

# new downloaded filenames (double underscores from the browser) — do NOT rename the files
TRAIN_PATH  = "Train.csv"
TEST_PATH   = "Test.csv"
SAMPLE_PATH = "SampleSubmission.csv"
NODATA = -9999            # satellite no-data sentinel in the new test set

In [2]:
# ============================================================
# Shared feature engineering — identical for train & test.
# Handles the -9999 no-data values via NaN-aware statistics.
# ============================================================
MONTHS = [f"{i:02d}" for i in range(1, 13)]
BANDS  = ["VH","VV","blue","green","nir","nira","re1","re2","re3","red","swir1","swir2"]

def load_raw(path):
    """Load a CSV and convert the -9999 no-data sentinel to NaN."""
    return pd.read_csv(path).replace(NODATA, np.nan)

def safe_ratio(a, b):
    # NaN propagates naturally; +0 denominator -> inf, cleaned later
    return (a - b) / (a + b)

def build_features(df):
    """Raw monthly bands -> engineered features.
    NaN-aware: every summary is computed over the VALID months only.
    Adds *_nvalid = how many months actually had data (a quality signal)."""
    feats = {}; idx = {}
    for m in MONTHS:
        g = df[f"green_{m}"]; n = df[f"nir_{m}"]; r = df[f"red_{m}"]; s1 = df[f"swir1_{m}"]
        idx.setdefault("NDWI",  []).append(safe_ratio(g,  n))
        idx.setdefault("MNDWI", []).append(safe_ratio(g,  s1))
        idx.setdefault("NDVI",  []).append(safe_ratio(n,  r))
        idx.setdefault("NDMI",  []).append(safe_ratio(n,  s1))
        idx.setdefault("VHmVV", []).append(df[f"VH_{m}"] - df[f"VV_{m}"])
    def summarize(monthly, name):
        a = np.column_stack([np.asarray(x, float) for x in monthly])  # rows x 12
        feats[f"{name}_mean"]  = np.nanmean(a, 1)
        feats[f"{name}_std"]   = np.nanstd(a, 1)
        feats[f"{name}_min"]   = np.nanmin(a, 1)
        feats[f"{name}_max"]   = np.nanmax(a, 1)
        feats[f"{name}_range"] = np.nanmax(a, 1) - np.nanmin(a, 1)
        feats[f"{name}_med"]   = np.nanmedian(a, 1)
        feats[f"{name}_nvalid"]= np.isfinite(a).sum(1)     # data-quality feature
    for b in BANDS: summarize([df[f"{b}_{m}"] for m in MONTHS], b)
    for k, v in idx.items(): summarize(v, k)
    return pd.DataFrame(feats, index=df.index).replace([np.inf, -np.inf], np.nan)

def corrupt(df, rate, seed=0):
    """Randomly blank band cells to mimic the test set's missingness."""
    rng = np.random.RandomState(seed); d = df.copy()
    bc = [c for c in d.columns if c not in ("ID", "label")]
    arr = d[bc].values.astype(float)
    arr[rng.rand(*arr.shape) < rate] = np.nan
    d[bc] = arr
    return d

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

train = load_raw(TRAIN_PATH); test = load_raw(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
y = train["label"].values

# augment with simulated missingness, then fit ensemble on ALL data
parts = [train] + [corrupt(train, r, seed=i+1) for i, r in enumerate((0.3, 0.6))]
aug = pd.concat(parts, ignore_index=True); yaug = np.concatenate([y]*len(parts))
X_aug = build_features(aug); X_test = build_features(test)

hgb = HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05,
        max_leaf_nodes=7, l2_regularization=1.0, random_state=42).fit(X_aug, yaug)
lr  = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                    LogisticRegression(max_iter=2000)).fit(X_aug, yaug)
proba = 0.5*hgb.predict_proba(X_test)[:,1] + 0.5*lr.predict_proba(X_test)[:,1]
print("test proba range:", round(proba.min(),3), "-", round(proba.max(),3))

test proba range: 0.0 - 0.999


In [4]:
submission = pd.DataFrame({
    "ID": test["ID"],
    "TargetF1": (proba >= 0.5).astype(int),   # locked 0.5 threshold
    "TargetRAUC": proba,
})
checks = {
  "columns match": list(submission.columns)==list(sample.columns),
  "row count": len(submission)==len(sample),
  "IDs match": set(submission.ID)==set(sample.ID),
  "TargetF1 0/1": set(submission.TargetF1.unique()).issubset({0,1}),
  "RAUC in [0,1]": submission.TargetRAUC.between(0,1).all(),
  "no NaN": submission.notna().all().all(),
}
for k,v in checks.items(): print(("PASS" if v else "FAIL"),"-",k)
assert all(checks.values())
print(f"\npredicted ponds: {submission.TargetF1.sum()} ({submission.TargetF1.mean():.1%}) vs train {y.mean():.1%}")
submission.to_csv("submission.csv", index=False)
print("wrote submission.csv")

PASS - columns match
PASS - row count
PASS - IDs match
PASS - TargetF1 0/1
PASS - RAUC in [0,1]
PASS - no NaN

predicted ponds: 565 (54.9%) vs train 40.4%
wrote submission.csv


## Trustworthiness sections (draft — rewritten for the new data)

### 1. Data & Model Bias
- **Data-quality bias (dominant):** the test set has ~60% missing (`-9999`) values while training is clean — a severe quality mismatch. We treated `-9999` as missing, summarized over valid months only, and **simulated the missingness during training** so the model isn't biased toward dense-observation locations.
- **Coordinate removal:** the new release dropped lon/lat, so we could no longer test geographic generalization directly; we used feature-space clustering as a proxy, which is weaker and may under-detect purely spatial bias.
- **Prevalence bias:** training is ~40% ponds; the test set's true rate is unknown, so the fixed-0.5 F1 remains sensitive to that gap.

### 2. Model Transparency
- **SHAP** (TreeExplainer) on the augmented gradient booster plus logistic-coefficient cross-check.
- Water/vegetation index summaries (NDWI/NDVI/MNDWI) dominate — physically sensible.
- **Dataset-specific finding:** the engineered **`*_nvalid` (months-with-data)** features carry real importance, meaning *how much* a location was observed is itself informative for land-cover here.

### 3. Approach Reusability
- `build_features` is sensor-generic and now **missingness-robust** by construction — directly reusable for other satellite tasks where cloud gaps are the norm.
- The **corruption-augmentation** trick generalizes to any setting with a clean-train / degraded-test mismatch.
- Limitation: fixed 12-month structure; irregular series need the summary step generalized.

### 4. Sustainability & Efficiency
- Tiny CPU models; even with 3× augmented data, training is seconds — no GPU. Estimate with **CodeCarbon** (`EmissionsTracker`); emissions are well under a gram CO₂e.
- Efficiency choices: compact engineered features over deep learning; augmentation adds modest cost for a large robustness gain — a favorable trade.

In [ ]:
# --- optional: CodeCarbon ---
# from codecarbon import EmissionsTracker
# t = EmissionsTracker(save_to_file=False); t.start()
# HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, max_leaf_nodes=7,
#     l2_regularization=1.0, random_state=42).fit(X_aug, yaug)
# print(f"{t.stop()*1000:.3f} g CO2e")

## The whole rebuilt arc
Verify the new data (0) → EDA incl. missingness (1) → **NaN-aware** features + `*_nvalid` (2) → honest validation **without coords, with corrupted folds** (3) → baselines (4) → **train-time missingness augmentation + ensemble** (5) → SHAP confirms physics + data-quality signal (6) → validated, reproducible submission (7).

Honest CV ≈ **0.92**, matching the live leaderboard's range. Trust that number over the public score; submit, then improve only what `honest_cv` confirms.